# RAG Step 2: Embeddings and Vector Store
Turn each chunk into a vector and store it in ChromaDB.

In [ ]:
!pip install -q sentence-transformers chromadb

In [ ]:
import json
from pathlib import Path
# Reads chunks.json from Step 1; falls back to a tiny sample if it is missing
if Path("chunks.json").exists():
    records = json.load(open("chunks.json"))
else:
    records = [{"id": "demo-0", "text": "RAG combines retrieval with generation.", "metadata": {"source": "demo", "chunk": 0}}]
print(len(records), "chunks")

## 1. Embed with sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode([r["text"] for r in records], normalize_embeddings=True)
print(embeddings.shape)

## 2. Store in ChromaDB (persistent)

In [ ]:
import chromadb
client = chromadb.PersistentClient(path="./chroma_store")
col = client.get_or_create_collection("rag_demo", metadata={"hnsw:space": "cosine"})
col.upsert(
    ids=[r["id"] for r in records],
    documents=[r["text"] for r in records],
    metadatas=[r["metadata"] for r in records],
    embeddings=embeddings.tolist(),
)
print("stored:", col.count())

## 3. Sanity check

In [ ]:
q = model.encode(["What reduces hallucinations?"], normalize_embeddings=True).tolist()
res = col.query(query_embeddings=q, n_results=2)
for d, dist in zip(res["documents"][0], res["distances"][0]):
    print(f"{dist:.3f}  {d}")

## Takeaways
Use the same embedding model for indexing and querying. Next: retrieval.